# NumPy Dojo — Block 5: End-to-End MLP

Dataset: **sklearn digits** — 8×8 grayscale handwritten digits, 10 classes.

Goal: build a **1-hidden-layer MLP** from scratch using only NumPy:
- Forward pass
- Backpropagation
- Gradient verification via finite differences
- Mini-batch training loop with Adam
- Track train/val loss and accuracy

This is the most important notebook. If you can write this cold, you're ready.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# ─────────────────────────────────────────────
# Data setup — copy your standardize() and split_data() here
# or re-implement inline
# ─────────────────────────────────────────────
digits = load_digits()
X_raw = digits.data.astype(np.float64)  # (1797, 64)
y = digits.target                        # (1797,)

# Standardize
mean = X_raw.mean(axis=0, keepdims=True)
std  = X_raw.std(axis=0, keepdims=True)
std[std == 0] = 1.0
X = (X_raw - mean) / std

# Split
np.random.seed(0)
idx = np.random.permutation(len(X))
n_train = int(0.7 * len(X))
n_val   = int(0.15 * len(X))
X_train, y_train = X[idx[:n_train]], y[idx[:n_train]]
X_val,   y_val   = X[idx[n_train:n_train+n_val]], y[idx[n_train:n_train+n_val]]
X_test,  y_test  = X[idx[n_train+n_val:]], y[idx[n_train+n_val:]]

N, D = X_train.shape
C = 10   # classes
H = 128  # hidden units

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

---
## P13 — Weight Initialization

Initialize weights for a 1-hidden-layer MLP:  `D → H → C`

- Use **He initialization** for ReLU: `W ~ N(0, sqrt(2/fan_in))`
- Biases initialized to zero
- Pack everything into a dict `params`

In [ ]:
def init_params(D, H, C, seed=42):
    """
    Returns dict with keys: W1 (H,D), b1 (H,), W2 (C,H), b2 (C,)
    """
    np.random.seed(seed)
    # TODO: He init for W1 and W2
    # W ~ np.random.randn(...) * sqrt(2 / fan_in)
    # fan_in for W1 = D, for W2 = H
    pass

params = init_params(D, H, C)

In [ ]:
# --- ASSERTS ---
assert params['W1'].shape == (H, D)
assert params['b1'].shape == (H,)
assert params['W2'].shape == (C, H)
assert params['b2'].shape == (C,)

# He init: std of W1 should be ≈ sqrt(2/D)
expected_std_W1 = np.sqrt(2.0 / D)
actual_std_W1 = params['W1'].std()
assert abs(actual_std_W1 - expected_std_W1) < 0.02, f'W1 std={actual_std_W1:.4f}, expected≈{expected_std_W1:.4f}'

assert np.all(params['b1'] == 0)
assert np.all(params['b2'] == 0)
print('P13 PASSED ✓')

---
## P14 — Forward Pass

Architecture: `X → Linear → ReLU → Linear → Softmax`

Return both the loss and a `cache` dict with all intermediate values needed for backprop.

**Cache must contain:** `X, Z1, A1, Z2, probs` (you'll need them in backward)

In [ ]:
def relu(x):
    return np.maximum(0, x)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def forward(X, params):
    """
    X: (N, D)
    Returns: loss (scalar), cache (dict)
    """
    W1, b1 = params['W1'], params['b1']
    W2, b2 = params['W2'], params['b2']
    
    # TODO: Z1 = X @ W1.T + b1
    # TODO: A1 = relu(Z1)
    # TODO: Z2 = A1 @ W2.T + b2
    # TODO: probs = softmax(Z2)
    # TODO: loss = mean cross-entropy
    # Don't forget to pass labels — add y as a parameter
    pass

# Adjust signature if needed:
def forward(X, y, params):
    W1, b1 = params['W1'], params['b1']
    W2, b2 = params['W2'], params['b2']
    
    # Layer 1
    # TODO
    
    # Layer 2
    # TODO
    
    # Loss
    # TODO
    
    cache = {}  # TODO: fill in
    return loss, cache

In [ ]:
# --- ASSERTS ---
loss, cache = forward(X_train[:32], y_train[:32], params)

assert np.isscalar(loss) or loss.ndim == 0, 'Loss should be scalar'
assert loss > 0, 'Loss must be positive'

# With random init, loss should be near log(C) = log(10) ≈ 2.3
assert 1.5 < loss < 3.5, f'Initial loss should be near log(10)≈2.3, got {loss:.4f}'

# Cache contents
for key in ['X', 'Z1', 'A1', 'Z2', 'probs']:
    assert key in cache, f'Cache missing "{key}"'

assert cache['probs'].shape == (32, C)
assert np.allclose(cache['probs'].sum(axis=1), 1.0, atol=1e-5)
assert np.all(cache['A1'] >= 0), 'ReLU output should be non-negative'

print(f'P14 PASSED ✓  initial loss={loss:.4f}')

---
## P15 — Backward Pass

Derive and implement backpropagation for the MLP.

Gradient chain (work backwards):
```
dL/dZ2  = probs - one_hot(y)          [softmax + CE gradient combined]
dW2     = dZ2.T @ A1 / N
db2     = dZ2.mean(axis=0)
dA1     = dZ2 @ W2
dZ1     = dA1 * (Z1 > 0)              [ReLU gradient]
dW1     = dZ1.T @ X / N
db1     = dZ1.mean(axis=0)
```

Return a `grads` dict with keys: `W1, b1, W2, b2`

In [ ]:
def backward(y, params, cache):
    """
    y: (N,) integer labels
    Returns: grads dict with keys W1, b1, W2, b2
    """
    N = len(y)
    W2 = params['W2']
    
    probs = cache['probs']
    A1    = cache['A1']
    Z1    = cache['Z1']
    X     = cache['X']
    
    # dL/dZ2 — combined softmax + CE gradient
    # TODO: one-hot encode y, then (probs - one_hot) / N
    
    # dW2, db2
    # TODO
    
    # Backprop through ReLU
    # TODO: dA1 = dZ2 @ W2; dZ1 = dA1 * (Z1 > 0)
    
    # dW1, db1
    # TODO
    
    grads = {}
    return grads

In [ ]:
# --- GRADIENT CHECK (finite differences) ---
# This is a hard check — if it passes, your backprop is correct

def numerical_gradient(X, y, params, key, eps=1e-5):
    """Compute numerical gradient for params[key] using central differences."""
    W = params[key]
    grad_num = np.zeros_like(W)
    it = np.nditer(W, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        orig = W[idx]
        
        W[idx] = orig + eps
        loss_plus, _ = forward(X, y, params)
        
        W[idx] = orig - eps
        loss_minus, _ = forward(X, y, params)
        
        grad_num[idx] = (loss_plus - loss_minus) / (2 * eps)
        W[idx] = orig
        it.iternext()
    return grad_num

# Use tiny params for speed
D_s, H_s, C_s = 8, 6, 4
params_small = init_params(D_s, H_s, C_s, seed=1)
X_s = X_train[:10, :D_s]
y_s = y_train[:10] % C_s

loss_s, cache_s = forward(X_s, y_s, params_small)
grads_s = backward(y_s, params_small, cache_s)

for key in ['W1', 'b1', 'W2', 'b2']:
    grad_analytic = grads_s[key]
    grad_numeric  = numerical_gradient(X_s, y_s, params_small, key)
    
    rel_error = np.abs(grad_analytic - grad_numeric).max() / (np.abs(grad_numeric).max() + 1e-8)
    status = '✓' if rel_error < 1e-4 else '✗  ← FAIL'
    print(f'  {key}: rel_error={rel_error:.2e}  {status}')

print('\nP15: gradient check done. All rel_errors should be < 1e-4.')

---
## P16 — Training Loop

Wire forward + backward into a mini-batch training loop with Adam.

Track train loss, val loss, train accuracy, val accuracy per epoch.

In [ ]:
def accuracy(X, y, params):
    loss, cache = forward(X, y, params)
    preds = cache['probs'].argmax(axis=1)
    return (preds == y).mean()

def train(X_train, y_train, X_val, y_val, params,
          epochs=30, batch_size=64, lr=1e-3):
    """
    Returns: history dict with 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """
    # Initialize Adam state for each parameter
    # TODO: m and v dicts, same keys as params, initialized to zeros
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    t = 0  # global step counter for Adam
    
    for epoch in range(epochs):
        # Shuffle training data
        # TODO
        
        # Mini-batch loop
        for start in range(0, len(X_train), batch_size):
            X_batch = X_train[start:start+batch_size]
            y_batch = y_train[start:start+batch_size]
            
            t += 1
            
            # TODO: forward
            # TODO: backward
            # TODO: adam step for each parameter
            pass
        
        # Record metrics
        train_loss, _ = forward(X_train, y_train, params)
        val_loss,   _ = forward(X_val,   y_val,   params)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(accuracy(X_train, y_train, params))
        history['val_acc'].append(accuracy(X_val,   y_val,   params))
        
        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch+1:3d} | '
                  f'train_loss={history["train_loss"][-1]:.4f} | '
                  f'val_loss={history["val_loss"][-1]:.4f} | '
                  f'val_acc={history["val_acc"][-1]*100:.1f}%')
    
    return history

params = init_params(D, H, C)
history = train(X_train, y_train, X_val, y_val, params, epochs=50)

In [ ]:
# --- ASSERTS ---
final_val_acc = history['val_acc'][-1]
assert final_val_acc > 0.85, f'Val accuracy should exceed 85%, got {final_val_acc*100:.1f}%'

# Loss should decrease
assert history['train_loss'][-1] < history['train_loss'][0], 'Training loss did not decrease'

test_acc = accuracy(X_test, y_test, params)
print(f'\nP16 PASSED ✓')
print(f'  Final val accuracy:  {final_val_acc*100:.1f}%')
print(f'  Final test accuracy: {test_acc*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(np.array(history['train_acc']) * 100, label='Train')
axes[1].plot(np.array(history['val_acc'])   * 100, label='Val')
axes[1].set_title('Accuracy (%)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('NumPy MLP on Digits')
plt.tight_layout()
plt.show()

---
## P17 — Error Analysis

Inspect what the model gets wrong.

Tasks:
1. Compute the confusion matrix from scratch (no sklearn) — shape (C, C)
2. Plot it as a heatmap
3. Find the 5 test samples the model is most confident about but gets wrong

In [ ]:
def confusion_matrix(y_true, y_pred, num_classes):
    """
    Returns (C, C) confusion matrix.
    cm[i, j] = number of samples with true label i predicted as j
    """
    # TODO: initialize zeros matrix, fill in using fancy indexing or np.add.at
    pass

_, cache_test = forward(X_test, y_test, params)
preds_test = cache_test['probs'].argmax(axis=1)
probs_test = cache_test['probs']

cm = confusion_matrix(y_test, preds_test, C)

In [ ]:
# --- ASSERTS ---
assert cm.shape == (C, C)
assert cm.sum() == len(y_test), 'Confusion matrix should sum to test set size'
assert np.diag(cm).sum() == (preds_test == y_test).sum(), 'Diagonal should be correct preds'
print('P17 PASSED ✓')

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(cm, cmap='Blues')
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
for i in range(C):
    for j in range(C):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()*0.5 else 'black', fontsize=8)

# Top-5 confident wrong predictions
wrong_mask = preds_test != y_test
wrong_idxs = np.where(wrong_mask)[0]
confidence_wrong = probs_test[wrong_mask].max(axis=1)
top5_local = np.argsort(confidence_wrong)[-5:][::-1]
top5_global = wrong_idxs[top5_local]

from sklearn.datasets import load_digits
digits_test_images = load_digits().images[len(X_train)+len(X_val):][top5_global]

for k, (idx, conf) in enumerate(zip(top5_global, confidence_wrong[top5_local])):
    ax = axes[1] if k == 0 else plt.subplot(1, 5, k+1) if False else None

fig2, ax2s = plt.subplots(1, 5, figsize=(12, 2.5))
for k, (g_idx, conf) in enumerate(zip(top5_global, confidence_wrong[top5_local])):
    from sklearn.datasets import load_digits as ld
    all_images = ld().images
    all_idx = np.random.permutation(len(all_images))
    break

# Simpler approach
all_data = load_digits()
rng = np.random.RandomState(0)
perm = rng.permutation(len(all_data.images))
test_global_indices = perm[n_train + int(0.15*len(all_data.images)):]

fig2, ax2s = plt.subplots(1, min(5, len(top5_global)), figsize=(10, 2.5))
if len(top5_global) < 5:
    ax2s = [ax2s] if not hasattr(ax2s, '__len__') else ax2s
for k, (local_idx, conf) in enumerate(zip(top5_local, confidence_wrong[top5_local])):
    if k >= 5: break
    global_idx = wrong_idxs[local_idx]
    ax2s[k].imshow(X_test[global_idx].reshape(8, 8), cmap='gray')
    ax2s[k].set_title(f'True:{y_test[global_idx]}\nPred:{preds_test[global_idx]}\n{conf*100:.0f}%')
    ax2s[k].axis('off')
plt.suptitle('Most Confident Wrong Predictions')
plt.tight_layout()
plt.show()